<a href="https://colab.research.google.com/github/ishak21/-Streamlit-Chatbot/blob/main/Streamlit_for_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streamlit for Chatbot — Materi Training

Source repo: [adiptamartulandi/chatbot-streamlit-demo](https://github.com/adiptamartulandi/chatbot-streamlit-demo)

## Apa itu Streamlit?

**Streamlit** adalah framework Python open-source yang memungkinkan kamu membangun web app interaktif hanya dengan Python — tanpa perlu HTML, CSS, atau JavaScript sama sekali.

Cukup tulis skrip Python biasa, dan Streamlit akan otomatis mengubahnya menjadi tampilan web yang interaktif.

Ini sangat cocok untuk:
- Demo model machine learning
- Dashboard data
- Aplikasi chatbot berbasis LLM
- Prototipe cepat yang bisa langsung dibagikan ke orang lain

## Kenapa Streamlit untuk Chatbot?

| Pendekatan | Kompleksitas | Kecepatan build |
|---|---|---|
| Flask + HTML/JS | Tinggi | Lama |
| FastAPI + React | Sangat tinggi | Sangat lama |
| **Streamlit** | Rendah | Cepat (menit) |

Streamlit punya `st.chat_message()` dan `st.chat_input()` yang dirancang khusus untuk membangun antarmuka chatbot dengan sedikit baris code.

## Struktur Notebook Ini

Notebook ini dibagi menjadi tiga bagian:

1. **Setup** — install Streamlit dan konfigurasi ngrok untuk menjalankan di Google Colab
2. **Part 1: Komponen Dasar Streamlit** — pengenalan elemen-elemen UI yang tersedia
3. **Part 2: Chatbot dengan Gemini** — membangun chatbot lengkap dengan session state dan Gemini API

## Kenapa Pakai ngrok?

Google Colab tidak punya IP publik. Streamlit berjalan di port 8501 di dalam server Colab,
tapi kita tidak bisa mengaksesnya langsung dari browser.

**ngrok** membuat "tunnel" dari internet ke port tersebut — sehingga app kamu bisa diakses
via URL publik tanpa install apapun di komputer lokal.

```
Browser kamu
    ↕  (HTTPS)
  ngrok server  ←→  tunnel  ←→  Google Colab (port 8501)
                                       ↕
                               Streamlit app berjalan
```


## Setup: Install Library & Konfigurasi ngrok

### Step 1 — Install Streamlit dan pyngrok

In [ ]:
!pip install -q streamlit pyngrok google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 86.3 MB/s eta 0:00:00


### Step 2 — Daftarkan ngrok Auth Token

Cara mendapatkan token:
1. Daftar gratis di [https://ngrok.com](https://ngrok.com)
2. Login → klik **Your Authtoken** di dashboard
3. Copy token-nya, paste ke Colab Secrets dengan nama `NGROK_TOKEN`
   (klik ikon 🔑 di sidebar kiri Colab)

> Token ngrok gratis sudah cukup untuk keperluan training.
> Satu akun ngrok bisa membuka 1 tunnel aktif sekaligus.

In [22]:
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('ngrok'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


### Fungsi Helper: Jalankan Streamlit + Buka Tunnel

Kita buat fungsi reusable yang akan kita pakai berulang kali di setiap bagian.
Fungsi ini melakukan tiga hal:
1. Menulis file `.py` ke disk
2. Menjalankan `streamlit run` di background
3. Membuka tunnel ngrok dan menampilkan URL publik

In [ ]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

---

# Part 1: Komponen Dasar Streamlit

Sebelum membangun chatbot, kita pelajari dulu elemen-elemen UI yang tersedia di Streamlit.
Sumber: `streamlit_app_basic.py` dari repo demo.

## Cara Kerja Streamlit

Streamlit bekerja dengan model yang unik: **setiap kali user berinteraksi dengan UI
(klik tombol, geser slider, ketik input), seluruh skrip Python dijalankan ulang dari atas ke bawah.**

Ini berbeda dengan framework web tradisional. Karena itulah kita nanti butuh `st.session_state`
untuk menyimpan data yang tidak boleh hilang saat skrip dijalankan ulang.

## Elemen-elemen UI Utama

### Teks & Judul
- `st.title()` — judul halaman (paling besar)
- `st.header()` — sub-judul
- `st.subheader()` — sub-sub-judul
- `st.write()` — teks biasa, bisa juga menerima dataframe, chart, dll
- `st.markdown()` — teks dengan format Markdown

### Input dari User
- `st.text_input()` — kotak teks satu baris
- `st.text_area()` — kotak teks multi baris
- `st.button()` — tombol klik
- `st.checkbox()` — toggle on/off
- `st.selectbox()` — dropdown pilihan
- `st.slider()` — slider angka
- `st.file_uploader()` — upload file

### Layout
- `st.sidebar` — panel samping yang bisa disembunyikan
- `st.columns()` — bagi halaman jadi beberapa kolom
- `st.expander()` — section yang bisa dilipat/dibuka

### Notifikasi
- `st.success()` — pesan hijau (berhasil)
- `st.info()` — pesan biru (informasi)
- `st.warning()` — pesan kuning (peringatan)
- `st.error()` — pesan merah (error)

### Data & Chart
- `st.dataframe()` — tabel interaktif
- `st.line_chart()`, `st.bar_chart()`, `st.area_chart()` — chart bawaan Streamlit
- `st.pyplot()` — tampilkan chart Matplotlib

Mari kita tulis dan jalankan file demo-nya:

In [ ]:
%%writefile streamlit_app_basic.py
import streamlit as st
import pandas as pd
import numpy as np
import time

# ── Judul & Intro ────────────────────────────────────────────────────────────
st.title("Streamlit Basic Tutorial")
st.write("""
## Selamat datang di Streamlit!

Ini adalah demo komponen-komponen dasar yang tersedia di Streamlit.
Setiap section di bawah menunjukkan cara kerja satu komponen.
""")

# ── 1. Text Input ────────────────────────────────────────────────────────────
st.header("1. Text Input")
st.write("""`st.text_input()` membuat kotak teks. Nilai yang diketik user disimpan di variabel.""")
user_input = st.text_input("Masukkan namamu", "Ketik di sini...")
st.write(f"Halo, {user_input}!")

# ── 2. Button ────────────────────────────────────────────────────────────────
st.header("2. Button")
st.write("""`st.button()` mengembalikan `True` saat diklik (lalu kembali ke `False`).""")
if st.button("Klik aku!"):
    st.write("Tombol diklik!")

# ── 3. Checkbox ──────────────────────────────────────────────────────────────
st.header("3. Checkbox")
st.write("""`st.checkbox()` adalah toggle yang mengembalikan `True` saat dicentang.""")
show_content = st.checkbox("Tampilkan pesan rahasia")
if show_content:
    st.write("Kamu menemukan pesan rahasianya!")

# ── 4. Selectbox ─────────────────────────────────────────────────────────────
st.header("4. Selectbox")
st.write("""`st.selectbox()` membuat dropdown untuk memilih satu opsi.""")
option = st.selectbox("Pilih warna favoritmu", ("Merah", "Biru", "Hijau", "Kuning"))
st.write(f"Kamu memilih: {option}")

# ── 5. Slider ────────────────────────────────────────────────────────────────
st.header("5. Slider")
st.write("""`st.slider()` membuat slider untuk memilih nilai angka secara interaktif.""")
age = st.slider("Berapa umurmu?", 0, 100, 25)
st.write(f"Umurmu adalah {age} tahun")

# ── 6. Progress Bar ──────────────────────────────────────────────────────────
st.header("6. Progress Bar")
st.write("""`st.progress()` menampilkan bar progress yang bisa diupdate.""")
progress_bar = st.progress(0)
for i in range(100):
    time.sleep(0.01)
    progress_bar.progress(i + 1)
st.write("Selesai!")

# ── 7. Sidebar ───────────────────────────────────────────────────────────────
st.header("7. Sidebar")
st.write("""`st.sidebar` membuat panel samping yang bisa disembunyikan.""")
with st.sidebar:
    st.header("Panel Samping")
    st.write("Widget di sini tidak mengganggu konten utama.")
    if st.button("Tombol di Sidebar"):
        st.write("Tombol sidebar diklik!")

# ── 8. Columns ───────────────────────────────────────────────────────────────
st.header("8. Columns")
st.write("""`st.columns()` membagi halaman menjadi beberapa kolom.""")
col1, col2 = st.columns(2)
with col1:
    st.write("Ini kolom kiri")
    st.button("Tombol di kolom kiri")
with col2:
    st.write("Ini kolom kanan")
    st.button("Tombol di kolom kanan")

# ── 9. Status Messages ───────────────────────────────────────────────────────
st.header("9. Status Messages")
st.success("Ini pesan sukses!")
st.info("Ini pesan informasi")
st.warning("Ini pesan peringatan")
st.error("Ini pesan error")

# ── 10. Charts ───────────────────────────────────────────────────────────────
st.header("10. Charts")

st.subheader("Line Chart")
chart_data = pd.DataFrame(
    np.random.randn(20, 3),
    columns=["Metrik A", "Metrik B", "Metrik C"]
)
st.line_chart(chart_data)

st.subheader("Bar Chart")
bar_data = pd.DataFrame(
    {"Apel": [10, 25, 18, 30], "Mangga": [15, 12, 22, 8]},
    index=["Jan", "Feb", "Mar", "Apr"]
)
st.bar_chart(bar_data)

# ── 11. Dataframe ────────────────────────────────────────────────────────────
st.header("11. Dataframe & Tabel")
data = {
    "Nama":  ["Alice", "Bob", "Charlie", "Diana"],
    "Skor":  [88, 72, 95, 80],
    "Level": ["A", "B", "A+", "A"],
}
df = pd.DataFrame(data)
st.write("Tabel interaktif (`st.dataframe`):")
st.dataframe(df)
st.write("Statistik deskriptif:")
st.write(df.describe())


Overwriting streamlit_app_basic.py


Setelah file ditulis, jalankan dengan ngrok:

In [ ]:
proc = run_streamlit("streamlit_app_basic.py")

Streamlit berjalan!
Buka URL ini di browser: NgrokTunnel: "https://wrinkly-outmatch-glutinous.ngrok-free.dev" -> "http://localhost:8501"


### Yang Perlu Diperhatikan

Coba berinteraksi dengan semua komponen di app yang baru terbuka. Perhatikan bahwa:

1. **Setiap interaksi menyebabkan skrip jalan ulang** — Progress bar akan berputar lagi setiap kali kamu klik sesuatu. Ini adalah perilaku normal Streamlit.
2. **Sidebar bisa disembunyikan** — klik panah kecil di pojok kiri atas app.
3. **Chart otomatis responsif** — coba resize jendela browser.

Untuk menghentikan app dan menjalankan app berikutnya, jalankan cell di bawah:

In [ ]:
# Hentikan app sebelumnya sebelum menjalankan app baru
import os, signal
try:
    proc.terminate()
    print("App dihentikan.")
except:
    pass

App dihentikan.


---

# Part 2: Chatbot dengan Gemini

Sekarang kita masuk ke bagian utama — membangun chatbot berbasis LLM menggunakan Streamlit + Gemini API.
Sumber: `streamlit_chat_app.py` dari repo demo.

## Konsep Kunci: st.session_state

Ini adalah konsep paling penting untuk memahami cara kerja chatbot di Streamlit.

Ingat bahwa Streamlit menjalankan ulang seluruh skrip setiap kali ada interaksi.
Ini artinya semua variabel Python akan di-reset ke nilai awal setiap kali user kirim pesan.

**Masalah:** Kalau pesan-pesan chat disimpan di variabel biasa, semua pesan akan hilang setiap kali user mengetik sesuatu!

**Solusi:** `st.session_state` adalah dictionary khusus yang **nilainya tetap tersimpan** meskipun skrip dijalankan ulang.

```python
# Variabel biasa — akan HILANG setiap rerun
messages = []

# st.session_state — akan BERTAHAN setiap rerun  
st.session_state.messages = []
```

Di chatbot kita, `st.session_state` digunakan untuk menyimpan:
- `messages` — riwayat semua pesan (user + assistant)
- `chat` — objek chat session dari Gemini API (menyimpan konteks percakapan)
- `genai_client` — client Gemini yang sudah terinisialisasi
- `_last_key` — API key terakhir yang dipakai (untuk deteksi perubahan key)

## Komponen Chat Khusus Streamlit

```python
# Menampilkan bubble chat — role bisa "user" atau "assistant"
with st.chat_message("user"):
    st.markdown("pesan dari user")

with st.chat_message("assistant"):
    st.markdown("jawaban dari AI")

# Kotak input di bagian bawah halaman
prompt = st.chat_input("Ketik pesanmu...")
```

## Alur Logic Chatbot

```
Skrip dijalankan (setiap interaksi)
         ↓
Cek API key di sidebar
         ↓
Inisialisasi client & chat session (hanya kalau belum ada di session_state)
         ↓
Tampilkan semua pesan dari st.session_state.messages
         ↓
Tunggu input dari st.chat_input()
         ↓
User kirim pesan?
   ↓ Ya
Tambah pesan user ke messages
Tampilkan bubble user
Kirim ke Gemini API
Tampilkan bubble assistant
Tambah jawaban ke messages
   ↓
Kembali ke atas (skrip dijalankan ulang)
```

## Cara Mendapatkan Google AI API Key

1. Buka [https://aistudio.google.com](https://aistudio.google.com)
2. Klik **Get API Key** → **Create API Key**
3. Copy key-nya
4. Di app Streamlit nanti, paste langsung ke kotak "Google AI API Key" di sidebar

> Key tidak perlu disimpan di Colab Secrets — user memasukkannya langsung di UI app.

In [ ]:
%%writefile streamlit_chat_app.py
# Import library yang dibutuhkan
import streamlit as st          # framework web app
from google import genai         # SDK Gemini dari Google

# ── 1. Konfigurasi Halaman ───────────────────────────────────────────────────
# st.title() dan st.caption() menampilkan judul dan keterangan di bagian atas
st.title("Gemini Chatbot")
st.caption("Chatbot sederhana menggunakan Google Gemini Flash")

# ── 2. Sidebar: Pengaturan App ───────────────────────────────────────────────
# Semua widget di dalam blok 'with st.sidebar:' akan muncul di panel samping
with st.sidebar:
    st.subheader("Pengaturan")

    # Kotak input untuk API key
    # type="password" menyembunyikan teks yang diketik (muncul sebagai titik-titik)
    google_api_key = st.text_input("Google AI API Key", type="password")

    # Tombol untuk mereset percakapan
    # Parameter 'help' menampilkan tooltip saat kursor diarahkan ke tombol
    reset_button = st.button("Reset Percakapan", help="Hapus semua pesan dan mulai dari awal")

# ── 3. Validasi API Key ──────────────────────────────────────────────────────
# Kalau user belum memasukkan API key, tampilkan pesan dan hentikan eksekusi
if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    # st.stop() menghentikan eksekusi skrip di titik ini
    # Kode setelah st.stop() tidak akan dijalankan
    st.stop()

# ── 4. Inisialisasi Gemini Client ────────────────────────────────────────────
# Bagian ini hanya membuat client baru kalau:
# - Client belum pernah dibuat (pertama kali app dijalankan), ATAU
# - User mengganti API key di sidebar
#
# Kenapa perlu dicek seperti ini?
# Karena setiap interaksi user menyebabkan seluruh skrip dijalankan ulang.
# Tanpa pengecekan ini, kita akan membuat client baru setiap kali user ketik pesan
# — yang artinya konteks percakapan akan hilang terus.
#
# getattr(obj, 'attr', default) = cara aman mengakses atribut yang mungkin belum ada
if ("genai_client" not in st.session_state) or (
    getattr(st.session_state, "_last_key", None) != google_api_key
):
    try:
        # Buat client Gemini baru dengan API key dari sidebar
        st.session_state.genai_client = genai.Client(api_key=google_api_key)

        # Simpan key yang dipakai — untuk deteksi perubahan key nanti
        st.session_state._last_key = google_api_key

        # Kalau key berganti, hapus session chat lama
        # .pop() menghapus key dari dict dengan aman (tidak error kalau key tidak ada)
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)

    except Exception as e:
        st.error(f"API Key tidak valid: {e}")
        st.stop()

# ── 5. Inisialisasi Chat Session & Riwayat Pesan ────────────────────────────
# Inisialisasi chat session Gemini kalau belum ada
if "chat" not in st.session_state:
    # Buat chat session baru dengan model gemini-2.5-flash
    # Session ini menyimpan konteks percakapan di sisi Gemini
    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash"
    )

# Inisialisasi list riwayat pesan kalau belum ada
# List ini menyimpan semua pesan untuk ditampilkan kembali saat skrip dijalankan ulang
if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 6. Tombol Reset ──────────────────────────────────────────────────────────
# Kalau tombol reset diklik, hapus chat session dan riwayat pesan
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    # st.rerun() memaksa Streamlit me-refresh halaman dari awal
    # Ini akan menjalankan ulang seluruh skrip, dan karena session sudah dihapus,
    # chat baru akan dibuat di langkah 5
    st.rerun()

# ── 7. Tampilkan Riwayat Percakapan ─────────────────────────────────────────
# Loop ini menampilkan semua pesan yang sudah ada di session_state
# Setiap kali skrip dijalankan ulang, pesan-pesan ini ditampilkan kembali
for msg in st.session_state.messages:
    # st.chat_message() membuat bubble chat dengan role yang sesuai
    # role "user" = bubble di kanan, role "assistant" = bubble di kiri
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── 8. Input & Respons ───────────────────────────────────────────────────────
# st.chat_input() membuat kotak input di bagian bawah halaman
# Nilai yang diketik user tersimpan di variabel 'prompt'
# Variabel ini bernilai None kalau user belum mengirim pesan
prompt = st.chat_input("Ketik pesanmu di sini...")

# Hanya jalankan bagian ini kalau user mengirim pesan
if prompt:
    # Langkah 1: Tambah pesan user ke riwayat
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Langkah 2: Tampilkan bubble pesan user
    with st.chat_message("user"):
        st.markdown(prompt)

    # Langkah 3: Kirim ke Gemini dan tampilkan respons
    try:
        # Kirim pesan ke Gemini melalui chat session yang sudah ada
        # chat.send_message() secara otomatis menyertakan riwayat percakapan sebelumnya
        response = st.session_state.chat.send_message(prompt)

        # Ambil teks dari respons
        # hasattr() memeriksa apakah objek punya atribut tertentu
        # Ini mencegah error kalau format respons tidak terduga
        if hasattr(response, "text"):
            answer = response.text
        else:
            answer = str(response)

    except Exception as e:
        # Kalau ada error (misal: rate limit, koneksi putus), tampilkan pesan error
        answer = f"Terjadi error: {e}"

    # Langkah 4: Tampilkan bubble respons assistant
    with st.chat_message("assistant"):
        st.markdown(answer)

    # Langkah 5: Simpan respons ke riwayat
    st.session_state.messages.append({"role": "assistant", "content": answer})


Overwriting streamlit_chat_app.py


Jalankan chatbot-nya:

In [23]:
proc = run_streamlit("streamlit_chat_app.py")

Streamlit berjalan: NgrokTunnel: "https://b332-34-74-191-176.ngrok-free.app" -> "http://localhost:8501"


### Panduan Mencoba App

1. Buka URL yang muncul di atas
2. Di sidebar, masukkan Google AI API Key kamu
3. Mulai chat di kotak input bagian bawah
4. Coba tanya beberapa pertanyaan — perhatikan bahwa model **mengingat konteks** percakapan sebelumnya
5. Klik **Reset Percakapan** di sidebar — konteks akan hilang dan chat mulai dari awal

### Yang Bisa Dikustomisasi

Coba modifikasi file `streamlit_chat_app.py` untuk:

- **Ganti model** — ubah `"gemini-2.5-flash"` ke `"gemini-2.5-flash-lite"` atau model lain
- **Tambah system prompt** — saat membuat chat session, tambahkan:
  ```python
  st.session_state.chat = st.session_state.genai_client.chats.create(
      model="gemini-2.5-flash",
      config={"system_instruction": "Kamu adalah asisten yang selalu menjawab dalam Bahasa Indonesia"}
  )
  ```
- **Tambah parameter** di sidebar — misalnya slider untuk temperature

---

## Menghentikan App

Jalankan cell di bawah untuk menghentikan Streamlit dan menutup tunnel ngrok.

In [ ]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.


## Ringkasan

| Konsep | Penjelasan |
|---|---|
| `%%writefile` | Magic command Colab untuk menulis konten cell ke file di disk |
| `subprocess.Popen` | Menjalankan proses (Streamlit) di background tanpa memblokir notebook |
| `ngrok.connect()` | Membuat URL publik yang mengarah ke port lokal |
| `st.session_state` | Dictionary yang nilainya bertahan meskipun skrip dijalankan ulang |
| `st.chat_message()` | Membuat bubble chat dengan role user/assistant |
| `st.chat_input()` | Kotak input yang muncul di bagian bawah halaman |
| `st.stop()` | Menghentikan eksekusi skrip di titik tersebut |
| `st.rerun()` | Memaksa Streamlit me-refresh halaman dari awal |

## Referensi

- [Dokumentasi Streamlit](https://docs.streamlit.io)
- [Streamlit Chat Elements](https://docs.streamlit.io/develop/api-reference/chat)
- [Session State](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state)
- [Repo Demo](https://github.com/adiptamartulandi/chatbot-streamlit-demo)
